# 00 — Rodando o SQL direto no Python (sem precisar do DB Browser)
**Projeto:** churn-upsell-analytics

Este notebook roda as 4 queries do arquivo `sql/01_sql_exploration.sql`
usando só Python — sem precisar de nenhum programa extra instalado.

**Por que isso funciona:** o Python tem uma biblioteca chamada `sqlite3`
que já vem instalada por padrão (não precisa dar pip install). Ela cria
um banco de dados temporário na memória do computador, e conseguimos
"conversar" com esse banco usando SQL de verdade — o mesmo SQL que está
no arquivo `.sql` do projeto.


## 1. Imports e carregar o CSV para dentro de um banco SQL

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Ajuste este caminho se o notebook estiver em uma pasta diferente
PROJECT_ROOT = Path.cwd().parent
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "Churn_Modelling.csv"

# sqlite3.connect(":memory:") cria um banco de dados que existe só na
# memória RAM enquanto o notebook está aberto — não cria nenhum arquivo
# .db no seu computador, e desaparece quando você reinicia o kernel
conn = sqlite3.connect(":memory:")

# Carrega o CSV para um DataFrame do Pandas (a "tabela" na memória do Python)
df_sql = pd.read_csv(CSV_PATH)

# .to_sql() envia esse DataFrame para dentro do banco SQLite,
# criando automaticamente uma tabela chamada "customers"
# if_exists="replace" garante que, se você rodar essa célula de novo,
# a tabela é recriada do zero (sem duplicar dados)
df_sql.to_sql("customers", conn, index=False, if_exists="replace")

print(f"Tabela 'customers' criada com {len(df_sql):,} linhas")
print(f"Colunas: {df_sql.columns.tolist()}")

## 2. Query 1 — Segmentação de risco de churn (CTE + CASE WHEN)

**Ponto de atenção importante:** a query SQL inteira precisa estar
dentro de `""" ... """` (três aspas duplas no início e três no
final). Isso é o que se chama de **string multi-linha** em Python —
sem isso, o Python tenta interpretar cada linha do SQL como se fosse
código Python, e quebra com `SyntaxError`, porque `WITH`, `SELECT`,
`FROM` não são comandos Python.


In [ ]:
query1 = """
WITH customer_segments AS (
    SELECT
        CustomerId,
        Geography,
        Age,
        Balance,
        NumOfProducts,
        IsActiveMember,
        Exited,
        CASE
            WHEN Age >= 50 THEN 'Senior (50+)'
            WHEN Age >= 35 THEN 'Adult (35-49)'
            ELSE 'Young (18-34)'
        END AS age_segment,
        CASE
            WHEN Balance = 0 THEN 'Zero Balance'
            WHEN Balance < 50000 THEN 'Low Balance'
            WHEN Balance < 150000 THEN 'Mid Balance'
            ELSE 'High Balance'
        END AS balance_segment
    FROM customers
)
SELECT
    age_segment,
    balance_segment,
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 1) AS churn_rate_pct
FROM customer_segments
GROUP BY age_segment, balance_segment
ORDER BY churn_rate_pct DESC
LIMIT 10;
"""

# pd.read_sql executa a string SQL dentro do banco 'conn' e já devolve
# o resultado como uma tabela do Pandas, prontinha para visualizar
resultado1 = pd.read_sql(query1, conn)
print("=== QUERY 1: Segmentos de maior risco de churn ===")
print(resultado1.to_string(index=False))

## 3. Query 2 — Ranking por saldo dentro de cada país (Window Functions)

In [ ]:
query2 = """
WITH ranked_customers AS (
    SELECT
        CustomerId,
        Geography,
        Balance,
        EstimatedSalary,
        Exited,
        RANK() OVER (PARTITION BY Geography ORDER BY Balance DESC) AS balance_rank,
        AVG(Balance) OVER (PARTITION BY Geography) AS avg_balance_geo,
        Balance - AVG(Balance) OVER (PARTITION BY Geography) AS balance_vs_avg
    FROM customers
)
SELECT
    Geography, CustomerId, Balance, balance_rank,
    ROUND(avg_balance_geo, 2) AS avg_balance_geo,
    ROUND(balance_vs_avg, 2)  AS balance_vs_avg
FROM ranked_customers
WHERE balance_rank <= 3
ORDER BY Geography, balance_rank;
"""

resultado2 = pd.read_sql(query2, conn)
print("=== QUERY 2: Top 3 clientes por saldo em cada país ===")
print(resultado2.to_string(index=False))

## 4. Query 3 — Scoring de upsell (NTILE + CTE aninhada)

In [ ]:
query3 = """
WITH upsell_base AS (
    SELECT
        CustomerId,
        Geography,
        NumOfProducts,
        IsActiveMember,
        Balance,
        EstimatedSalary,
        Exited,
        NTILE(4) OVER (ORDER BY Balance DESC) AS balance_quartile
    FROM customers
    WHERE Exited = 0
),
scored AS (
    SELECT
        *,
        CASE
            WHEN NumOfProducts = 1 AND IsActiveMember = 1 AND balance_quartile <= 2
                THEN 'High Upsell Potential'
            WHEN NumOfProducts = 1 AND IsActiveMember = 1
                THEN 'Medium Upsell Potential'
            ELSE 'Low Upsell Potential'
        END AS upsell_segment
    FROM upsell_base
)
SELECT
    upsell_segment,
    COUNT(*) AS customers,
    ROUND(AVG(Balance), 2) AS avg_balance,
    ROUND(AVG(EstimatedSalary), 2) AS avg_salary
FROM scored
GROUP BY upsell_segment
ORDER BY customers DESC;
"""

resultado3 = pd.read_sql(query3, conn)
print("=== QUERY 3: Segmentos de propensão a upsell ===")
print(resultado3.to_string(index=False))

## 5. Query 4 — Churn por número de produtos

In [ ]:
query4 = """
SELECT
    NumOfProducts,
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned,
    ROUND(100.0 * SUM(Exited) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY NumOfProducts
ORDER BY NumOfProducts;
"""

resultado4 = pd.read_sql(query4, conn)
print("=== QUERY 4: Churn por número de produtos ===")
print(resultado4.to_string(index=False))

if resultado4["churn_rate_pct"].iloc[2:].mean() > resultado4["churn_rate_pct"].iloc[:2].mean():
    print()
    print("⚠ Achado confirmado: clientes com 3-4 produtos têm MAIOR churn")
    print("  que clientes com 1-2 produtos — padrão contraintuitivo real.")

## 6. Conclusão

Se todas as 4 queries acima rodaram e mostraram tabelas de resultado,
o SQL do projeto está 100% funcional. Você pode:

1. Tirar screenshots dos resultados (Windows: tecla Print Screen, ou a
   ferramenta "Recorte" do Windows) para colocar em `docs/screenshots/`
2. Considerar este notebook como prova de que o SQL roda de verdade —
   pode até incluí-lo no repositório como `notebooks/00_sql_runner.ipynb`
